[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/naylaartha2-png/TUGAS-UAS-STATISTIKA/blob/main/Nayla_Artha_Mevia_Suma_F5212510009.ipynb)

# Prediksi Risiko Penduduk Terkena Tindak Pidana Berdasarkan Jumlah Kejahatan, Persentase Penyelesaian, dan Selang Waktu Terjadinya Kejahatan Menurut Provinsi di Indonesia Tahun 2017–2021 Menggunakan Regresi Linear Berganda



#  Data Collecting

Dataset yang digunakan merupakan data kejahatan menurut provinsi di Indonesia dari tahun 2017 hingga 2021 berdasarkan data dari BPS (Badan Pusat Statistik). Dataset ini memuat informasi mengenai jumlah kejahatan, risiko penduduk terkena tindak pidana per 100.000 penduduk, persentase penyelesaian kejahatan, dan selang waktu terjadinya kejahatan di setiap provinsi.

**Variabel yang digunakan:**
- **Variabel Target (Y):** `Risiko_per_100rb` — Risiko penduduk terkena tindak pidana per 100.000 penduduk
- **Variabel Prediktor (X):**
  - `Jml_Kejahatan` — Jumlah kejahatan yang terjadi
  - `Pct_Selesai` — Persentase penyelesaian perkara kejahatan (%)
  - `Selang_Waktu` — Selang waktu terjadinya kejahatan (hari)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

print("Library berhasil diimport!")

In [ ]:
# Membaca dataset
# Pastikan file 'data_kejahatan_bersih.csv' sudah diupload ke Google Colab
file_path = "data_kejahatan_bersih.csv"
df = pd.read_csv(file_path)

print("Dataset berhasil dimuat!")

#  Exploratory Data Analysis (EDA)

Pada tahap ini, kita akan mengeksplorasi dataset untuk memahami struktur data, distribusi variabel, dan hubungan antar variabel sebelum membangun model prediksi.

In [ ]:
# Menampilkan 5 data awal untuk memastikan data terbaca
print("=== 5 Data Pertama ===")
df.head()

In [ ]:
# Statistik deskriptif
print("=== Statistik Deskriptif ===")
df.describe()

In [ ]:
# Informasi tipe data dan jumlah data
print("=== Informasi Dataset ===")
df.info()

In [ ]:
# Ukuran dataset
print(f"Jumlah baris (observasi) : {df.shape[0]}")
print(f"Jumlah kolom (variabel)  : {df.shape[1]}")
print(f"Jumlah provinsi          : {df['Provinsi'].nunique()}")
print(f"Rentang tahun            : {df['Tahun'].min()} - {df['Tahun'].max()}")

In [ ]:
# Mengecek nilai kosong (missing values)
print("=== Jumlah Nilai Kosong per Kolom ===")
print(df.isnull().sum())

In [ ]:
# Mengecek data duplikat
print(f"Jumlah data duplikat: {df.duplicated().sum()}")

In [ ]:
# DETEKSI DAN VISUALISASI OUTLIER (PENCILAN)
# Menggunakan metode IQR (Interquartile Range)

kolom_target = 'Risiko_per_100rb'

print(f"==== Deteksi Outlier dengan IQR pada kolom {kolom_target} ====")

Q1 = df[kolom_target].quantile(0.25)
Q3 = df[kolom_target].quantile(0.75)
IQR = Q3 - Q1

# Menentukan batas bawah dan batas atas
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Mencari data yang berada di luar batas (outlier)
outliers = df[(df[kolom_target] < lower_bound) | (df[kolom_target] > upper_bound)]
print(f"Batas Bawah : {lower_bound}")
print(f"Batas Atas  : {upper_bound}")
print(f"Jumlah {kolom_target} Outlier : {len(outliers)} baris dari total {len(df)} baris data")

# Visualisasi Outlier dengan Boxplot
plt.figure(figsize=(8, 4))
sns.boxplot(x=df[kolom_target], color='lightcoral')
plt.title(f'Visualisasi Outlier pada Kolom {kolom_target} (Boxplot)', fontsize=13)
plt.xlabel('Risiko per 100.000 Penduduk')
plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi distribusi semua variabel numerik
numerik_cols = ['Jml_Kejahatan', 'Risiko_per_100rb', 'Pct_Selesai', 'Selang_Waktu']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Distribusi Variabel Numerik', fontsize=14, fontweight='bold')

for ax, col in zip(axes.flatten(), numerik_cols):
    sns.histplot(df[col], kde=True, ax=ax, color='steelblue')
    ax.set_title(f'Distribusi {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Frekuensi')

plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi korelasi antar variabel
plt.figure(figsize=(8, 6))
corr_matrix = df[numerik_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title('Matriks Korelasi Variabel', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n=== Nilai Korelasi terhadap Risiko_per_100rb ===")
print(corr_matrix['Risiko_per_100rb'].sort_values(ascending=False))

In [ ]:
# Tren rata-rata risiko kejahatan per tahun
plt.figure(figsize=(10, 5))
trend = df.groupby('Tahun')['Risiko_per_100rb'].mean().reset_index()
plt.plot(trend['Tahun'], trend['Risiko_per_100rb'], marker='o', color='crimson', linewidth=2)
plt.title('Tren Rata-Rata Risiko Kejahatan per 100.000 Penduduk (2017–2021)', fontsize=13)
plt.xlabel('Tahun')
plt.ylabel('Rata-rata Risiko per 100.000 Penduduk')
plt.xticks(trend['Tahun'].astype(int))
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [2]:
# Top 10 provinsi dengan rata-rata risiko kejahatan tertinggi
plt.figure(figsize=(10, 6))
top10 = df.groupby('Provinsi')['Risiko_per_100rb'].mean().sort_values(ascending=False).head(10)
sns.barplot(x=top10.values, y=top10.index, palette='Reds_r')
plt.title('Top 10 Provinsi dengan Rata-rata Risiko Kejahatan Tertinggi', fontsize=13)
plt.xlabel('Rata-rata Risiko per 100.000 Penduduk')
plt.ylabel('Provinsi')
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

#  Data Preprocessing

Pada tahap ini dilakukan persiapan data sebelum dimasukkan ke dalam model, meliputi penanganan outlier, pemilihan fitur, dan pembagian data menjadi data latih dan data uji.

In [ ]:
print(f"Jumlah data sebelum pembersihan outlier: {len(df)}")

# Menghapus outlier pada variabel target (Risiko_per_100rb)
df_clean = df[(df['Risiko_per_100rb'] >= lower_bound) & (df['Risiko_per_100rb'] <= upper_bound)].copy()

print(f"Jumlah data setelah pembersihan outlier: {len(df_clean)}")
print(f"Data yang dihapus: {len(df) - len(df_clean)} baris")

In [ ]:
# Memilih kolom yang digunakan untuk analisis
# X = Variabel prediktor (independen)
# y = Variabel target (dependen)

X = df_clean[['Jml_Kejahatan', 'Pct_Selesai', 'Selang_Waktu']]
y = df_clean['Risiko_per_100rb']

print("=== Variabel Prediktor (X) ===")
print(X.head())
print(f"\nShape X: {X.shape}")
print(f"\n=== Variabel Target (y) ===")
print(y.head())

In [ ]:
# Standardisasi fitur menggunakan StandardScaler
# Agar semua fitur berada dalam skala yang sama
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("=== Data Setelah Standardisasi ===")
print(X_scaled.describe().round(2))

In [ ]:
# Membagi data menjadi data latih (train) dan data uji (test)
# 80% data latih dan 20% data uji
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"Jumlah data latih (train) : {X_train.shape[0]} baris")
print(f"Jumlah data uji  (test)   : {X_test.shape[0]} baris")

#  Model Training

Pada tahap ini, model **Regresi Linear Berganda** dilatih menggunakan data latih. Regresi linear berganda digunakan untuk memodelkan hubungan antara satu variabel dependen (risiko kejahatan) dengan dua atau lebih variabel independen (jumlah kejahatan, persentase penyelesaian, dan selang waktu).

Persamaan umum regresi linear berganda:

$$Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \beta_3 X_3 + \varepsilon$$

Di mana:
- $Y$ = Risiko per 100.000 penduduk
- $X_1$ = Jumlah Kejahatan
- $X_2$ = Persentase Penyelesaian
- $X_3$ = Selang Waktu
- $\beta_0$ = Intercept (konstanta)
- $\beta_1, \beta_2, \beta_3$ = Koefisien regresi

In [ ]:
# Membuat dan melatih model Regresi Linear Berganda
model = LinearRegression()
model.fit(X_train, y_train)

# Menampilkan koefisien model
print("=== PERSAMAAN REGRESI LINEAR BERGANDA ===")
print(f"Intercept (β₀) : {model.intercept_:.4f}")
print()
for nama, koef in zip(X.columns, model.coef_):
    print(f"Koefisien {nama} : {koef:.4f}")

# Melakukan prediksi pada data uji
y_pred = model.predict(X_test)

In [ ]:
# Menampilkan persamaan regresi secara lengkap
b0 = model.intercept_
b1, b2, b3 = model.coef_

print("=== PERSAMAAN MODEL ===")
print(f"Risiko_per_100rb = {b0:.4f}")
print(f"                 + ({b1:.4f}) × Jml_Kejahatan")
print(f"                 + ({b2:.4f}) × Pct_Selesai")
print(f"                 + ({b3:.4f}) × Selang_Waktu")

#  Visualisasi Hasil Prediksi

In [ ]:
# Mengatur gaya plot
sns.set_theme(style="whitegrid")

# VISUALISASI 1: Nilai Aktual vs Nilai Prediksi (Scatter Plot)
# Semakin titik-titik mendekati garis merah diagonal, semakin akurat prediksi model
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.6, color='steelblue', edgecolors='white', linewidth=0.5)

# Garis referensi diagonal (prediksi sempurna)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Prediksi Sempurna')

plt.title('Nilai Aktual vs Nilai Prediksi Risiko Kejahatan', fontsize=14)
plt.xlabel('Risiko Aktual (per 100.000 penduduk)', fontsize=12)
plt.ylabel('Risiko Prediksi (per 100.000 penduduk)', fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# VISUALISASI 2: Residual Plot
# Residual = selisih antara nilai aktual dan nilai prediksi
# Residual yang tersebar merata di sekitar nol menunjukkan model yang baik
residual = y_test.values - y_pred

plt.figure(figsize=(10, 5))
plt.scatter(y_pred, residual, alpha=0.6, color='orange', edgecolors='white', linewidth=0.5)
plt.axhline(y=0, color='red', linestyle='--', linewidth=1.5)
plt.title('Plot Residual (Nilai Prediksi vs Residual)', fontsize=13)
plt.xlabel('Nilai Prediksi', fontsize=11)
plt.ylabel('Residual', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# VISUALISASI 3: Distribusi Residual
# Distribusi residual yang mendekati normal menunjukkan asumsi linearitas terpenuhi
plt.figure(figsize=(8, 5))
sns.histplot(residual, kde=True, color='purple', bins=20)
plt.axvline(x=0, color='red', linestyle='--', linewidth=1.5)
plt.title('Distribusi Residual', fontsize=13)
plt.xlabel('Residual')
plt.ylabel('Frekuensi')
plt.tight_layout()
plt.show()

In [3]:
# VISUALISASI 4: Koefisien Regresi (Feature Importance)
koef_df = pd.DataFrame({
    'Variabel': X.columns,
    'Koefisien': model.coef_
}).sort_values('Koefisien', ascending=True)

plt.figure(figsize=(8, 4))
colors = ['crimson' if k < 0 else 'seagreen' for k in koef_df['Koefisien']]
plt.barh(koef_df['Variabel'], koef_df['Koefisien'], color=colors)
plt.axvline(x=0, color='black', linewidth=0.8, linestyle='--')
plt.title('Koefisien Regresi Linear Berganda', fontsize=13)
plt.xlabel('Nilai Koefisien')
plt.tight_layout()
plt.show()

print("Interpretasi:")
print("- Koefisien positif (hijau) → variabel meningkatkan risiko kejahatan")
print("- Koefisien negatif (merah) → variabel menurunkan risiko kejahatan")

NameError: name 'pd' is not defined

#  Evaluasi Model

Evaluasi model dilakukan untuk mengukur seberapa baik model dalam memprediksi risiko kejahatan. Metrik yang digunakan:

| Metrik | Keterangan |
| ------ | ----------- |
| MAE    | Mean Absolute Error — rata-rata kesalahan absolut antara nilai aktual dan prediksi |
| MSE    | Mean Squared Error — rata-rata kuadrat kesalahan; memberi hukuman lebih pada error besar |
| RMSE   | Root Mean Squared Error — akar dari MSE; mudah diinterpretasikan karena satuannya sama dengan target |
| R²     | Koefisien determinasi — proporsi variansi target yang dapat dijelaskan oleh model (0–1, semakin mendekati 1 semakin baik) |

In [ ]:
# Menghitung metrik evaluasi model
r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("\n=== EVALUASI MODEL REGRESI LINEAR BERGANDA ===")
print(f"R²   (Koefisien Determinasi) : {r2:.4f}  ({r2*100:.2f}%)")
print(f"MAE  (Mean Absolute Error)   : {mae:.4f}")
print(f"MSE  (Mean Squared Error)    : {mse:.4f}")
print(f"RMSE (Root MSE)              : {rmse:.4f}")

print("\n=== INTERPRETASI ===")
if r2 >= 0.7:
    print(f"Model memiliki R² = {r2:.2f}, artinya {r2*100:.1f}% variasi risiko kejahatan")
    print("dapat dijelaskan oleh variabel prediktor yang digunakan. Model CUKUP BAIK.")
elif r2 >= 0.5:
    print(f"Model memiliki R² = {r2:.2f}, artinya {r2*100:.1f}% variasi risiko kejahatan")
    print("dapat dijelaskan oleh variabel prediktor. Model CUKUP MODERAT.")
else:
    print(f"Model memiliki R² = {r2:.2f}, artinya {r2*100:.1f}% variasi risiko kejahatan")
    print("dapat dijelaskan oleh variabel prediktor. Perlu penambahan fitur atau transformasi data.")

In [ ]:
# Menampilkan contoh hasil prediksi vs nilai aktual
hasil_uji = pd.DataFrame({
    'Risiko Aktual': y_test.values[:10],
    'Risiko Prediksi': np.round(y_pred[:10], 2),
    'Selisih (Error)': np.abs(y_test.values[:10] - y_pred[:10]).round(2)
})

print("=== CONTOH HASIL PREDIKSI (10 Data Pertama) ===")
print(hasil_uji.to_string(index=False))

#  Deployment / Prediksi Data Baru

Pada tahap ini, model yang telah dilatih digunakan untuk memprediksi risiko kejahatan berdasarkan input data baru dari pengguna.

In [ ]:
# ============================================================
# PREDIKSI RISIKO KEJAHATAN BERDASARKAN INPUT PENGGUNA
# ============================================================

print("=== SISTEM PREDIKSI RISIKO KEJAHATAN ===")
print("Masukkan data berikut untuk memprediksi risiko penduduk terkena tindak pidana:")
print()

# Contoh input data baru (simulasi)
# Untuk penggunaan interaktif di Colab, ubah nilai di bawah ini
jml_kejahatan = 7000    # Jumlah kejahatan
pct_selesai   = 65.0    # Persentase penyelesaian perkara (%)
selang_waktu  = 100.0   # Selang waktu kejahatan (hari)

print(f"Input Data Baru:")
print(f"  Jumlah Kejahatan        : {jml_kejahatan}")
print(f"  Persentase Penyelesaian : {pct_selesai}%")
print(f"  Selang Waktu Kejahatan  : {selang_waktu} hari")

# Menyiapkan data baru ke dalam DataFrame
data_baru = pd.DataFrame([{
    'Jml_Kejahatan': jml_kejahatan,
    'Pct_Selesai'  : pct_selesai,
    'Selang_Waktu' : selang_waktu
}])

# Standardisasi data baru menggunakan scaler yang sudah di-fit
data_baru_scaled = scaler.transform(data_baru)

# Melakukan prediksi
prediksi = model.predict(data_baru_scaled)[0]

print()
print("=" * 45)
print(f"HASIL PREDIKSI:")
print(f"  Risiko per 100.000 Penduduk : {prediksi:.2f} orang")
print("=" * 45)

# Interpretasi hasil
rata_nasional = df['Risiko_per_100rb'].mean()
print(f"\nRata-rata nasional (2017-2021) : {rata_nasional:.2f} orang per 100.000 penduduk")

if prediksi > rata_nasional:
    print(f"Prediksi MELEBIHI rata-rata nasional → Risiko TINGGI")
else:
    print(f"Prediksi DI BAWAH rata-rata nasional → Risiko RENDAH")

In [ ]:
# ============================================================
# KESIMPULAN
# ============================================================

print("=" * 55)
print("KESIMPULAN ANALISIS REGRESI LINEAR BERGANDA")
print("=" * 55)
print()
print("Judul:")
print("  Prediksi Risiko Penduduk Terkena Tindak Pidana")
print("  Menurut Provinsi di Indonesia Tahun 2017–2021")
print()
print("Variabel Prediktor (X):")
print("  1. Jumlah Kejahatan (Jml_Kejahatan)")
print("  2. Persentase Penyelesaian Perkara (Pct_Selesai)")
print("  3. Selang Waktu Terjadinya Kejahatan (Selang_Waktu)")
print()
print("Variabel Target (Y):")
print("  Risiko per 100.000 Penduduk (Risiko_per_100rb)")
print()
print("Persamaan Model:")
print(f"  Y = {b0:.4f} + ({b1:.4f})×X₁ + ({b2:.4f})×X₂ + ({b3:.4f})×X₃")
print()
print("Hasil Evaluasi:")
print(f"  R²   = {r2:.4f}  → Model menjelaskan {r2*100:.2f}% variansi data")
print(f"  MAE  = {mae:.4f}  → Rata-rata kesalahan absolut")
print(f"  RMSE = {rmse:.4f}  → Ukuran error yang mudah diinterpretasikan")
print()
print("=" * 55)